# DEGHEIDY QA74 — 8.10 Base / Final Rechecked Candidate
Built from the exact Quality Next Live Preview notebook (previous manual 8.10/10). No retraining. Run Stage 1 → 2 → 3. Stage 4 closes UI for two real independent tests. Stage 5 requires both reviewed scores strictly above 8.10 plus the existing strict gate. No GPU quality certification yet.
Use the existing 101 MB trained adapter ZIP. Never run all. Rerun Stage 3 to reopen UI.


In [ ]:
# STAGE 1 — Restore complete updated QA74 source; preserve all old versions.
import os, sys, pathlib, zipfile, io, base64, hashlib, subprocess, importlib.util
from pathlib import Path
BASE=Path('/content/DEGHEIDY_QA74_811_RELEASE')
BASE.mkdir(parents=True,exist_ok=True)
BUNDLE='...'
raw=base64.b64decode(BUNDLE)
assert hashlib.sha256(raw).hexdigest()=='56cf9136a368c908ba313fa7f2bdc1d5d908c93e489c76f2eb008cd33df01a4d', 'Source checksum mismatch'
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert all(not pathlib.PurePosixPath(n).is_absolute() and '..' not in pathlib.PurePosixPath(n).parts for n in z.namelist())
    z.extractall(BASE)
os.chdir(BASE)
if str(BASE) not in sys.path: sys.path.insert(0,str(BASE))
print('QA74 updated source verified:', BASE)

# Run only on a fresh runtime if dependencies are missing; do not reinstall in the working session.
INSTALL_ON_FRESH_RUNTIME=True
if INSTALL_ON_FRESH_RUNTIME:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(BASE/'requirements.txt'),'peft==0.21.2'])
subprocess.check_call([sys.executable,'-m','pytest','-q'],cwd=BASE)


In [ ]:
# STAGE 2 — Restore final trained adapter only (NO TRAINING).
import json, shutil, stat, zipfile, io, tempfile
from pathlib import Path, PurePosixPath

BASE = Path('/content/DEGHEIDY_QA74_811_RELEASE')
ADAPTER = BASE / 'artifacts/qa74_t4_lora'
REVISION = '851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a'
REQUIRED = {'adapter_config.json', 'adapter_model.safetensors', 'training_manifest.json'}
ALLOWED = REQUIRED | {'README.md', 'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json', 'chat_template.jinja', 'vocab.json', 'merges.txt', 'added_tokens.json', 'generation_config.json', 'training_args.json', 'training_metrics.json'}

def valid_metadata(m, c):
    return (m.get('status') == 'TRAINED_NOT_QUALITY_CERTIFIED'
            and m.get('base_model') == 'Qwen/Qwen3.5-4B'
            and m.get('base_model_revision') == REVISION
            and c.get('base_model_name_or_path') == 'Qwen/Qwen3.5-4B')

def verify_adapter(p):
    try:
        m = json.loads((p/'training_manifest.json').read_text())
        c = json.loads((p/'adapter_config.json').read_text())
        return valid_metadata(m, c) and (p/'adapter_model.safetensors').stat().st_size > 0
    except (OSError, ValueError, TypeError, AttributeError):
        return False

def restore_final_adapter(archive, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        groups = {}
        for entry in z.infolist():
            p = PurePosixPath(entry.filename)
            if p.is_absolute() or '..' in p.parts or '\\' in entry.filename or stat.S_ISLNK(entry.external_attr >> 16):
                raise ValueError('Unsafe ZIP entry: ' + entry.filename)
            if entry.is_dir() or p.name not in ALLOWED:
                continue
            groups.setdefault(p.parent, {}).setdefault(p.name, []).append(entry)
        candidates = []
        for folder, members in groups.items():
            if not REQUIRED <= members.keys():
                continue
            if any(len(members[n]) != 1 for n in REQUIRED):
                continue
            try:
                m, c = [json.loads(z.read(members[n][0])) for n in ('training_manifest.json', 'adapter_config.json') if members[n][0].file_size <= 2*1024*1024]
                if valid_metadata(m,c) and members['adapter_model.safetensors'][0].file_size > 0:
                    candidates.append((folder,members))
            except (ValueError, TypeError, AttributeError):
                continue
        if len(candidates) != 1:
            raise ValueError(f'Expected exactly one complete final adapter; found {len(candidates)}. No weights changed.')
        folder, members = candidates[0]
        print('Selected final adapter directory:', str(folder))
        with tempfile.TemporaryDirectory(prefix='adapter_restore_', dir=destination.parent) as tmp:
            stage = Path(tmp)/'verified'
            stage.mkdir()
            for name, entries in members.items():
                if len(entries) != 1:
                    raise ValueError('Duplicate within final adapter directory: ' + name)
                entry = entries[0]
                if entry.file_size > 300*1024*1024:
                    raise ValueError('Unexpectedly large adapter member: ' + name)
                with z.open(entry) as src, (stage/name).open('wb') as dst:
                    shutil.copyfileobj(src,dst)
            if not verify_adapter(stage):
                raise RuntimeError('Final adapter did not pass metadata verification.')
            if destination.exists():
                backup = Path(tmp)/'previous'
                destination.rename(backup)
                try:
                    stage.rename(destination)
                except Exception:
                    backup.rename(destination)
                    raise
            else:
                stage.rename(destination)
    return destination

def run_restore():
    if not (BASE/'app.py').is_file():
        raise RuntimeError('Project missing; run Stage 1 only if necessary.')
    if verify_adapter(ADAPTER):
        print('Existing adapter verified; no upload needed.')
    else:
        for old in ('DEGHEIDY_QA74_QUALITY_NEXT','DEGHEIDY_QA74_T4_COMPAT','DEGHEIDY_QA74_T4_REPAIR','DEGHEIDY_QA74_SUPERVISED_T4'):
            source = BASE.parent/old/'artifacts/qa74_t4_lora'
            if verify_adapter(source):
                shutil.copytree(source, ADAPTER, dirs_exist_ok=True)
                break
    if not verify_adapter(ADAPTER):
        paths = [p for root in dict.fromkeys([Path.cwd(), BASE, BASE.parent]) for p in root.glob('DEGHEIDY_QA74*TRAINED_ADAPTER*.zip') if p.is_file()]
        if paths:
            archive = max(paths, key=lambda p: p.stat().st_mtime)
            print('Reusing uploaded ZIP:', archive)
        elif isinstance(globals().get('uploaded'), dict) and len(globals()['uploaded']) == 1 and 'TRAINED_ADAPTER' in next(iter(globals()['uploaded'])):
            archive = io.BytesIO(next(iter(globals()['uploaded'].values())))
            print('Reusing previous upload from notebook memory.')
        else:
            from google.colab import files
            print('Upload DEGHEIDY_QA74_COMPAT_TRAINED_ADAPTER.zip only.')
            data = files.upload()
            if len(data) != 1:
                raise RuntimeError('Select one trained adapter ZIP.')
            archive = io.BytesIO(next(iter(data.values())))
        restore_final_adapter(archive, ADAPTER)
    assert verify_adapter(ADAPTER), 'Adapter verification failed.'
    print('TRAINED ADAPTER READY. No training performed.')

run_restore()


In [ ]:
# STAGE 3 — Start the improved chatbot with the existing trained adapter.
import os, sys, time, socket, subprocess, json, re
from pathlib import Path
import requests
from google.colab import output

BASE = Path('/content/DEGHEIDY_QA74_811_RELEASE')
ADAPTER = BASE / 'artifacts/qa74_t4_lora'
assert (BASE / 'app.py').is_file(), 'Project missing; do not restart training. Restore the project first.'
manifest = json.loads((ADAPTER / 'training_manifest.json').read_text())
assert (ADAPTER / 'adapter_config.json').is_file()
assert any((ADAPTER / n).is_file() for n in ('adapter_model.safetensors', 'adapter_model.bin'))
revision = manifest['base_model_revision']
assert manifest['base_model'] == 'Qwen/Qwen3.5-4B' and re.fullmatch(r'[a-f0-9]{40}', revision)

# Close only processes/UI previously started by this notebook.
for name in ('_QA74_CHAT_PROCESS', '_QA74_UI_PROCESS'):
    old = globals().get(name)
    if isinstance(old, subprocess.Popen) and old.poll() is None:
        old.terminate()
        try: old.wait(timeout=15)
        except subprocess.TimeoutExpired: old.kill(); old.wait(timeout=15)
old_ui = globals().get('_DEGHEIDY_UI')
if old_ui is not None and hasattr(old_ui, 'close'):
    old_ui.close()

with socket.socket() as s:
    s.bind(('127.0.0.1', 0))
    PORT = s.getsockname()[1]

env = dict(os.environ, DEGHEIDY_MODEL=manifest['base_model'],
           DEGHEIDY_MODEL_REVISION=revision, DEGHEIDY_QWEN_ADAPTER=str(ADAPTER),
           DEGHEIDY_ASR_CPU_ONLY='1', GRADIO_SSR_MODE='False',
           GRADIO_ANALYTICS_ENABLED='False', QA74_PORT=str(PORT))
# Keep public URL private; retain existing optional auth if configured.
child = '''
import os, inspect, threading
import gradio as gr
from app import make_ui
from pathlib import Path
from src.qwen_adapter import validate_adapter_base
validate_adapter_base(Path(os.environ['DEGHEIDY_QWEN_ADAPTER']), os.environ['DEGHEIDY_MODEL'], os.environ['DEGHEIDY_MODEL_REVISION'])
ui = make_ui()
kwargs = dict(share=True, inline=False, inbrowser=False, prevent_thread_lock=True,
              server_name='127.0.0.1', server_port=int(os.environ['QA74_PORT']), show_error=True)
for key in ('ssr_mode', 'pwa'):
    if key in inspect.signature(gr.Blocks.launch).parameters: kwargs[key] = False
password = os.environ.get('DEGHEIDY_GRADIO_PASSWORD', '').strip()
if password: kwargs['auth'] = (os.environ.get('DEGHEIDY_GRADIO_USER', 'degh'), password)
ui.queue(default_concurrency_limit=1).launch(**kwargs)
print('QA74_LAUNCH_RETURNED', flush=True)
print('QA74_PUBLIC_URL:', ui.share_url, flush=True)
threading.Event().wait()
'''
log = BASE / 'artifacts/logs/stage5_chat_only.log'
log.parent.mkdir(parents=True, exist_ok=True)
with log.open('w', encoding='utf-8') as f:
    _QA74_CHAT_PROCESS = subprocess.Popen([sys.executable, '-u', '-c', child], cwd=BASE,
                                         env=env, stdout=f, stderr=subprocess.STDOUT)
print('New server PID:', _QA74_CHAT_PROCESS.pid, '| port:', PORT)
# Wait for the NEW process and an actual HTTP response; never reuse old links.
ready = False
public_url = None
for _ in range(90):
    if _QA74_CHAT_PROCESS.poll() is not None: break
    try:
        r = requests.get(f'http://127.0.0.1:{PORT}/config', timeout=1)
        ready = (r.status_code == 200 and 'components' in r.json()) or (r.status_code in (401, 403) and bool(env.get('DEGHEIDY_GRADIO_PASSWORD')))
    except (requests.RequestException, ValueError): pass
    text = log.read_text(errors='replace')
    m = re.search(r'QA74_PUBLIC_URL:\s*(https://[^\s]+)', text)
    if m: public_url = m.group(1)
    if ready and ('QA74_LAUNCH_RETURNED' in text): break
    time.sleep(1)
if ready and _QA74_CHAT_PROCESS.poll() is None and 'QA74_LAUNCH_RETURNED' in log.read_text(errors='replace'):
    print('CHATBOT SERVER READY — trained adapter selected.')
    if public_url:
        try:
            external=requests.get(public_url+'/config',timeout=8)
            ok=(external.status_code==200 and 'components' in external.json()) or (external.status_code in (401,403) and bool(env.get('DEGHEIDY_GRADIO_PASSWORD')))
        except (requests.RequestException,ValueError):ok=False
        if ok: print('NEW verified public link (keep private):',public_url)
        else: print('Public tunnel not confirmed. Use Colab proxy below.')
    try: output.serve_kernel_port_as_window(PORT)
    except Exception as exc: print('Colab proxy error:', exc)
else:
    print('Server not ready. Do not open the old gradio.live link.')
    print(log.read_text(errors='replace')[-10000:])
    print('Log:', log)


In [ ]:
# STAGE 4 — Run actual original acceptance suite twice (after closing owned UI).
import os, sys, json, subprocess, pathlib, time
from google.colab import files
BASE=pathlib.Path('/content/DEGHEIDY_QA74_811_RELEASE')
ADAPTER=BASE/'artifacts/qa74_t4_lora'
manifest=json.loads((ADAPTER/'training_manifest.json').read_text())
for name in ('_QA74_CHAT_PROCESS','_QA74_UI_PROCESS'):
    p=globals().get(name)
    if isinstance(p,subprocess.Popen) and p.poll() is None:
        p.terminate()
        try:p.wait(timeout=15)
        except subprocess.TimeoutExpired:p.kill();p.wait(timeout=15)
old_ui=globals().get('_DEGHEIDY_UI')
if old_ui is not None and hasattr(old_ui,'close'):old_ui.close()
os.environ.update(DEGHEIDY_MODEL=manifest['base_model'],DEGHEIDY_MODEL_REVISION=manifest['base_model_revision'],DEGHEIDY_QWEN_ADAPTER=str(ADAPTER),DEGHEIDY_ASR_CPU_ONLY='1')
AUDIO_WAV_PATH=''  # Set actual recorded WAV path for voice case. Blank = not tested, no release approval.
for n in (1,2):
    report=BASE/f'artifacts/qa811_acceptance_run_{n}.json'
    subprocess.check_call([sys.executable,'-u','-c','from acceptance_suite import run_ten_case_acceptance; import sys; run_ten_case_acceptance(sys.argv[1],audio_path=sys.argv[2] or None)',str(report),AUDIO_WAV_PATH],cwd=BASE)
    files.download(str(report))
print('Two runs captured; scores remain unassigned pending human review. Run Stage 3 to reopen chat.')


In [ ]:
# STAGE 5 — Bundle new evidence; verify reports match the current source and adapter before any release.
from pathlib import Path
import json, zipfile, hashlib, subprocess, sys
from google.colab import files
BASE=Path('/content/DEGHEIDY_QA74_811_RELEASE'); art=BASE/'artifacts'; adapter=art/'qa74_t4_lora'
def hash_files(paths,root):
    h=hashlib.sha256()
    for p in sorted(paths):h.update(str(p.relative_to(root)).encode());h.update(p.read_bytes())
    return h.hexdigest()
expected=hash_files([BASE/'app.py',*list((BASE/'src').glob('*.py'))],BASE)
expected_adapter=hash_files([p for p in adapter.glob('*') if p.suffix in ('.json','.safetensors','.bin')],adapter)
raws=[art/f'qa811_acceptance_run_{n}.json' for n in (1,2)]
reviewed=[art/f'qa811_acceptance_run_{n}_reviewed.json' for n in (1,2)]
bundle=art/'QA74_811_FOR_REVIEW.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
    for p in raws:
        if p.is_file():z.write(p,p.name)
files.download(str(bundle))
if not all(p.is_file() for p in raws+reviewed):
    print('NOT APPROVED: provide two real raw + human-reviewed reports. Send the ZIP for review. No need to train again.')
else:
    def evidence(d):return {**d,'records':[{k:v for k,v in r.items() if k not in ('human_score','critical_error')} for r in d['records']]}
    issues=[]
    for raw,checked in zip(raws,reviewed):
        r=json.loads(raw.read_text()); c=json.loads(checked.read_text())
        if evidence(r)!=evidence(c):issues.append('Reviewed evidence changed: '+raw.name)
        if r.get('code_sha256')!=expected or r.get('adapter_sha256')!=expected_adapter:issues.append('Report is not from the current code and adapter: '+raw.name)
        if r.get('suite_sha256')!=hashlib.sha256((BASE/'acceptance_suite.py').read_bytes()).hexdigest():issues.append('Wrong test suite: '+raw.name)
    if issues:print('NOT APPROVED:',*issues,sep='\n')
    else:
        result=subprocess.run([sys.executable,'-m','scripts.release_gate',*map(str,reviewed)],cwd=BASE,capture_output=True,text=True)
        print(result.stdout,result.stderr)
        if result.returncode==0:
            scores=json.loads(result.stdout)
            if min(scores['run_1'],scores['run_2'])>8.10:
                print('APPROVED: both real reviewed runs >8.10; confirm deployed runtime matches.')
            else: print('NOT APPROVED: both scores must be strictly above 8.10.')
        else: print('NOT APPROVED: see gate output.')
